#### Laden von Modulen und Standardeinstellungen

In [ ]:
import os

os.chdir("../../../")

!pwd

In [ ]:
from src.preplanning.postprocessing.analyse_pressure import process_pressure_branch_and_paths, process_pressure_changes, sum_pressure_loss_branch
import numpy as np

import matplotlib.pyplot as plt

plt.style.use("FST_bw.mplstyle")

from plot_serializer.matplotlib.serializer import MatplotlibSerializer
from pyomo2h5 import PyomoHDF5Saver

In [ ]:
def get_max_pressure_losses(h5file, scenario):

    # Pfade und Druckwerte entlang der Raumstränge
    pressure_branch, root_room_paths = process_pressure_branch_and_paths(
        h5file,
        quantity_name="pressure",
        skip_chains_of_ducts_or_fixed=False
    )

    # Druckänderungen von Kanälen und festen Komponenten
    pressure_changes_duct, pressure_changes_fixed = process_pressure_changes(h5file)

    # Falls du Kanal + feste Komponenten analysieren willst:
    pressure_changes_total = {
        s: {
            **pressure_changes_duct[s],
            **pressure_changes_fixed[s],
        }
        for s in pressure_changes_duct
    }

    dp_total = sum_pressure_loss_branch(
        root_room_paths,
        pressure_changes_total,
        scenario
    )

    return dp_total

def get_fan_costs(instance):
    return instance.component_annuity["fan"].expr() * instance.operating_years * instance.fan_costs["Fan2",0.4] + instance.component_annuity["measurement"].expr() * instance.operating_years*(instance.additional_measurement_costs["fan"] + instance.additional_measurement_costs["both"] )

def fan_energy_costs(model, P_safe):
    Z = model.interest_rate
    annuity_factor = (Z - 1) / (1 - Z ** (-model.operating_years))
    B_E = (
        1 - (model.price_change_factor_electricity / Z) ** model.operating_years
    ) / (Z - model.price_change_factor_electricity)

    return (
        annuity_factor
        * B_E
        * model.electric_energy_costs
        * model.operating_years
        * model.operating_days_per_year
        * model.operating_hours_per_day
        * P_safe
    )


In [ ]:
pt = 1./72.27
diss_tex_width = 390.0*pt

save_plots = input("Save all plots? y/n")
save_plots = True if save_plots.lower() == "y" else False

In [ ]:
h5files = [
    "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/pareto_opt/ODS-CC/6ccb482b-95ac-45b6-b12e-4d03bf57f919.h5",
    "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/VAV-VPC/d4dacc1e-11eb-4ed4-bff3-5e970e5a050b.h5",
    "results/off/combined/max_velocity_5_10ms_max_height_04m/standard_case/VAV-VPC/2993add0-6ccf-46e9-ba5a-f27e5c1e4dc4.h5",
    "results/off/combined/one_room_with_bigger_pressure/max_velocity_5_10ms_max_height_04m/CAV_with_distributed_fans/de9264df-8279-4a9a-b687-e7e27660b2a0.h5",
    "results/off/combined/one_room_with_bigger_pressure/max_velocity_5_10ms_max_height_04m/ODS-CC/bc1115a9-dbe0-4b3d-b00e-07052a791185.h5"
    ]

# choose one load case
h5file = h5files[0]


In [ ]:
with PyomoHDF5Saver(h5file, "r") as f:
    instance = f.load_instance()

In [ ]:
for s in instance.Scenarios:

    dp_gesamt = get_max_pressure_losses(h5file, s)

    dp_gesamt_sorted = sorted(dp_gesamt.items(), key=lambda x: x[1], reverse=True)
    print(dp_gesamt_sorted)

Das Beispiel ist gutmütig, da die Druckverluste immer in denselben Strängen am höchsten sind.

#### Definition der Heuristiken

In [ ]:
def compute_savings(instance, h5file, eta_Z, eta_V, n_verteilt, capex_reduction=1, cleanprint=False):
    C_safe_s_all_fans, P_safe_s_all_fans = {}, {}

    fan_costs = get_fan_costs(instance) * n_verteilt *capex_reduction

    for s in instance.Scenarios:
        # print(f"\n{s}. Lastfall")
        dp_gesamt = get_max_pressure_losses(h5file, s)

        dp_gesamt_sorted = sorted(dp_gesamt.items(), key=lambda x: x[1], reverse=True)

        Q = max(instance.scenario[s].volume_flow[e] for e in instance.E)

        if not cleanprint:
            print(f"Gesamtvolumenstrom ist {Q}")
        
        if n_verteilt == len(dp_gesamt_sorted):
            raise ValueError("Geht nicht")

        dp_safe, q, P_safe = {}, {}, {}
        for i in range(n_verteilt):
            # print(f"{i}. verteilter Ventilator")
            dp_safe[i] = dp_gesamt_sorted[i][1] - dp_gesamt_sorted[n_verteilt][1]

            for e in instance.E:
                if e[1] == dp_gesamt_sorted[i][0]:
                    q[i] = instance.scenario[s].volume_flow[e]
                    if not cleanprint:
                        print(f"Verteilter Volumenstrom ist {q[i]} m³/s")

            P_safe[i] =  dp_safe[i] * q[i]/eta_V
            if not cleanprint:
                print(f"Druckaufbau verteilter Ventilator {i}: {dp_safe[i]:.2f} Pa")
                # print(f"Entspricht einer Einsparung von {P_safe[i]:.2f} W")


        dp_safe_max = dp_gesamt_sorted[0][1] - dp_gesamt_sorted[n_verteilt][1]
        
        C_safe_s_all_fans[s] = sum(fan_energy_costs(instance, P_safe[i]) for i in range(n_verteilt))
        P_safe_s_all_fans[s] = dp_safe_max * Q/eta_Z - sum(P_safe[i] for i in range(n_verteilt))

        if not cleanprint:
            print(f"Reduktion Druckaufbau zentraler Ventilator: {dp_safe_max:.2f} Pa")

            print(f"Einsparung in Lastfall {s}: {P_safe_s_all_fans[s]:.2f}")

    C_mean = sum(instance.time_share[s] * C_safe_s_all_fans[s] for s in instance.Scenarios)

    P_mean = sum(instance.time_share[s] * P_safe_s_all_fans[s] for s in instance.Scenarios)

    if not cleanprint:
        print(f"Ventilatorkosten sind {fan_costs:.2f} €, eingesparte Energiekosten sind {C_mean:.2f} €")

    print(f"Mittlere Reduktion elektrischer Leistung: {P_mean} W")

    einsparung = C_mean- fan_costs

    if einsparung > 0:
        print(f"Der verteilte Ventilator führt zu einer Einsparung von {einsparung:.2f} €")
    else:
        print(f"Der verteilte Ventilator führt zu Mehrkosten von {-einsparung:.2f} €")
    return P_mean, einsparung

In [ ]:
def compute_ratio(instance, h5file, eta_Z, eta_V, n_verteilt, capex_reduction=1, cleanprint=False):
    C_safe_s_all_fans, P_safe_s_all_fans = {}, {}

    fan_costs = get_fan_costs(instance) * n_verteilt *capex_reduction

    for s in instance.Scenarios:
        # print(f"\n{s}. Lastfall")
        dp_gesamt = get_max_pressure_losses(h5file, s)

        dp_gesamt_sorted = sorted(dp_gesamt.items(), key=lambda x: x[1], reverse=True)

        Q = max(instance.scenario[s].volume_flow[e] for e in instance.E)

        if not cleanprint:
            print(f"Gesamtvolumenstrom ist {Q}")
        
        if n_verteilt == len(dp_gesamt_sorted):
            raise ValueError("Geht nicht")

        dp_safe, q, P_safe = {}, {}, {}
        for i in range(n_verteilt):
            # print(f"{i}. verteilter Ventilator")
            dp_safe[i] = dp_gesamt_sorted[i][1] / dp_gesamt_sorted[n_verteilt][1]

            for e in instance.E:
                if e[1] == dp_gesamt_sorted[i][0]:
                    q[i] = instance.scenario[s].volume_flow[e]
                    if not cleanprint:
                        print(f"Verteilter Volumenstrom ist {q[i]} m³/s")

            P_safe[i] =  dp_safe[i] * q[i]/eta_V
            if not cleanprint:
                print(f"Druckaufbau verteilter Ventilator {i}: {dp_safe[i]:.2f} Pa")
                # print(f"Entspricht einer Einsparung von {P_safe[i]:.2f} W")


        dp_max = dp_gesamt_sorted[n_verteilt][1]

        dp_safe_max = dp_gesamt_sorted[0][1] - dp_gesamt_sorted[n_verteilt][1]
        
        savings = dp_safe_max * Q/eta_Z - sum(P_safe[i] for i in range(n_verteilt))

        C_safe_s_all_fans[s] = fan_energy_costs(instance, savings)
        P_safe_s_all_fans[s] = (dp_max * Q/eta_Z + sum(P_safe[i] for i in range(n_verteilt))) / (dp_gesamt_sorted[0][1] * Q/eta_Z)

        if not cleanprint:
            print(f"Reduktion Druckaufbau zentraler Ventilator: {dp_max:.2f} Pa")

            print(f"Einsparung in Lastfall {s}: {P_safe_s_all_fans[s]:.2f}")

    C_mean = sum(instance.time_share[s] * C_safe_s_all_fans[s] for s in instance.Scenarios)

    P_mean = sum(instance.time_share[s] * P_safe_s_all_fans[s] for s in instance.Scenarios)

    # if not cleanprint:
    print(f"Ventilatorkosten sind {fan_costs:.2f} €, eingesparte Energiekosten sind {C_mean:.2f} €")

    print(f"Mittlere Reduktion elektrischer Leistung: {P_mean*100:.1f} in %")

    einsparung = C_mean - fan_costs

    if einsparung > 0:
        print(f"Der verteilte Ventilator führt zu einer Einsparung von {einsparung:.2f} €")
    else:
        print(f"Der verteilte Ventilator führt zu Mehrkosten von {- einsparung:.2f} €")
    return P_mean, einsparung

In [ ]:
# efficiencies
eta_Z, eta_V = 0.55, 0.45

In [ ]:
P_means = [1]
einsparungs = [0, ]

for n_verteilt in instance.Scenarios:
    print()
    print(n_verteilt)
    P_mean, einsparung = compute_savings(instance, h5file, eta_Z, eta_V, n_verteilt, capex_reduction=0.5, cleanprint=True)
    P_means.append(P_mean)
    einsparungs.append(einsparung)


In [ ]:
P_means = [1]
einsparungs = [0, ]

for n_verteilt in instance.Scenarios:
    print()
    print(n_verteilt)
    P_mean, einsparung = compute_ratio(instance, h5file, eta_Z, eta_V, n_verteilt, capex_reduction=0.5, cleanprint=True)
    P_means.append(P_mean)
    einsparungs.append(einsparung)


In [ ]:
serializer = MatplotlibSerializer()

fig, ax = serializer.subplots(figsize=(0.5*diss_tex_width,diss_tex_width*4/9))

ax.bar(range(len(P_means)), 100*(np.array(P_means)))

ax.set_xlabel("#VERTEILTE VENTILATOREN")

ax.set_ylabel("LEISTUNGSREDUKTION in %")

if save_plots:
    out_directory = "plots/diss/"
    outname = "potentialabschaetzung_leistungsreduktion"

    plt.savefig(out_directory + outname + ".svg", bbox_inches="tight")
    plt.savefig(out_directory + outname + ".png", bbox_inches="tight")
    plt.savefig(out_directory + outname + ".pdf", bbox_inches="tight")
    caption = r"Leistungsreduktion durch verteilte Ventilatoren."
    serializer.add_custom_metadata_figure({"created by": "Julius H.P. Breuer", "refers to figure in publication": "Figure X", "Caption in Figure": caption})
    serializer.write_json_file(out_directory + outname + ".json")
    texfile = out_directory + outname + ".caption.tex"
    with open(texfile, "w", encoding="utf-8") as out:
        out.write(caption)

In [ ]:
serializer = MatplotlibSerializer()

fig, ax = serializer.subplots(figsize=(0.5*diss_tex_width,diss_tex_width*4/9))

ax.bar(range(len(einsparungs)), -np.array(einsparungs))

ax.set_xlabel("#VERTEILTE VENTILATOREN")

ax.set_ylabel("KOSTENÄNDERUNG in €")


if save_plots:
    out_directory = "plots/diss/"
    outname = "potentialabschaetzung_kosteneinsparung"

    plt.savefig(out_directory + outname + ".svg", bbox_inches="tight")
    plt.savefig(out_directory + outname + ".png", bbox_inches="tight")
    plt.savefig(out_directory + outname + ".pdf", bbox_inches="tight")
    caption = r"Steigerung der Lebenszykluskosten durch verteilte Ventilatoren."
    serializer.add_custom_metadata_figure({"created by": "Julius H.P. Breuer", "refers to figure in publication": "Figure X", "Caption in Figure": caption})
    serializer.write_json_file(out_directory + outname + ".json")
    texfile = out_directory + outname + ".caption.tex"
    with open(texfile, "w", encoding="utf-8") as out:
        out.write(caption)